In [33]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, base64, os, glob
from PIL import Image
from io import BytesIO
import pytesseract
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
print("Good To Go")

Good To Go


In [18]:
# OPTIONAL: Set path to tesseract.exe if not in system PATH
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\admin\Desktop\OCR\tesseract_OCR\tesseract.exe"
print("done")

done


In [19]:
# Set your download directory (where you want your information is get downloaded)
download_dir = r"C:\Users\admin\Downloads\gem_pdf\all_pdf"
print("done")

done


In [20]:
# Set your browser where we are going to get information
#setup
chrome_options = Options()
# chrome_options.add_argument("--headless")  # Optional: for headless mode
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")  # Usually for Linux, safe to keep
## please ignore the commnet part
# Disable PDF viewer in Chrome
#chrome_options.add_experimental_option('excludeSwitches', ['enable-logging'])
#c
#hrome_options.add_argument('--disable-print-preview')
print("good to go")

good to go


In [21]:
# Set download behavior make sure the pdf get downloaded instead of opening
prefs = {
    "download.default_directory": download_dir,
    "plugins.always_open_pdf_externally": True,  # Download PDFs instead of opening in Chrome
}
chrome_options.add_experimental_option("prefs", prefs)
print("done")

done


In [22]:
# Read reference IDs from notepad file
with open(r"C:\Users\admin\Downloads\gem_pdf\ref.txt", "r") as f:
    content = f.read().strip()

# Split by comma and clean spaces
reference_ids = [ref.strip() for ref in content.split(",") if ref.strip()]

print("All Good")

Loaded reference IDs: ['GEM/2024/B/5588836', 'GEM/2025/B/6071338', 'GEM/2024/B/5594165', 'GEM/2024/B/5458607', 'GEM/2024/B/5480361', 'GEM/2024/B/4560359', 'GEM/2018/B/119037', 'GEM/2022/B/1919736', 'GEM/2024/B/5598504']


In [40]:
# Print total count
print("Total reference IDs:", len(reference_ids))

# Print each ID vertically
for ref in reference_ids:
    print(ref)

Total reference IDs: 9
GEM/2024/B/5588836
GEM/2025/B/6071338
GEM/2024/B/5594165
GEM/2024/B/5458607
GEM/2024/B/5480361
GEM/2024/B/4560359
GEM/2018/B/119037
GEM/2022/B/1919736
GEM/2024/B/5598504


In [23]:
base_url = 'https://gem.gov.in/view_contracts/bid_detail?bid_no='
print("site is ready")

site is ready


In [24]:
# Step 3: Set up Selenium WebDriver (e.g., Chrome)
driver = webdriver.Chrome(options=chrome_options)  # Update with the path to your chromedriver
print("open the site")

open the site


In [25]:
#wait report
time.sleep(5)
print("set")

set


In [26]:
# Store PDF data and show it after the information get downloaded
pdf_info_list = []

def wait_for_download_and_record(index, ref_id, download_dir):
    # Wait for .crdownload files to finish (max 20 seconds)
    for _ in range(20):
        cr_files = glob.glob(os.path.join(download_dir, "*.crdownload"))
        if not cr_files:
            break
        time.sleep(9)

    # Find the latest downloaded PDF
    pdf_files = glob.glob(os.path.join(download_dir, "*.pdf"))
    if not pdf_files:
        print(f"{ref_id} - PDF not found in folder!")
        return None

    latest_pdf = max(pdf_files, key=os.path.getctime)  # Get latest file
    original_pdf_name = os.path.basename(latest_pdf)

    print(f"{ref_id} - downloaded as {original_pdf_name}")

    return {
        "reference_id": ref_id,
        "pdf_name": original_pdf_name,
        "pdf_path": latest_pdf
    }
print("keep doing")

keep doing


In [27]:
# Loop through each reference ID
for index, ref_id in enumerate(reference_ids):
    full_url = base_url + ref_id
    driver.get(full_url)
    time.sleep(5)

    # Check if "No Result Found" exists
    try:
        no_result = driver.find_element(By.XPATH, "//div[contains(text(), 'No Result Found')]")
        print(f"{ref_id} - no result found")
        
        #  Append 'no result found' record
        pdf_info_list.append({
            "reference_id": ref_id,
            "pdf_name": "no result found",
            "pdf_path": "no result found"
        })
        
        continue
    except:
        pass  # No "No Result Found" message, proceed

    # Try to click to trigger captcha
    try:
        driver.find_element(By.CLASS_NAME, "ajxtag_order_number").click()
        time.sleep(5)
    except:
        print(f"{ref_id} - couldn't find contract number to click")
        continue

    # Loop to solve captcha
    for attempt in range(8):  # Try max 5 times
        try:
            # Get captcha image
            img_elem = driver.find_element(By.ID, "captchaimg")
            img_base64 = img_elem.get_attribute("src").split(",")[1]

            image = Image.open(BytesIO(base64.b64decode(img_base64)))
            captcha_text = pytesseract.image_to_string(image).strip()

            # Fill captcha
            driver.find_element(By.ID, "captcha_code").clear()
            driver.find_element(By.ID, "captcha_code").send_keys(captcha_text)

            # Submit
            driver.find_element(By.ID, "modelsbt").click()

            # Wait for download button
            wait = WebDriverWait(driver, 5)
            download_link = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#dwnbtn a")))
            download_link.click()

            print(f"{ref_id} - successfully downloaded")
            
            time.sleep(3)
            
            # ✅ Append download info
            pdf_info = wait_for_download_and_record(index, ref_id, download_dir)
            if pdf_info:
                pdf_info_list.append(pdf_info)
            else:
                #  No PDF found even after clicking download
                pdf_info_list.append({
                    "reference_id": ref_id,
                    "pdf_name": "no result found",
                    "pdf_path": "no result found"
                })
            
            time.sleep(6)
            break

        except:
            print(f"{ref_id} - captcha attempt {attempt + 1} failed, retrying...")
            try:
                refresh_btn = driver.find_element(By.XPATH, "//img[@src='/resources/images/refresh.png']")
                refresh_btn.click()
                time.sleep(6)
            except:
                print(f"{ref_id} - captcha refresh failed")
                break
    else:
        print(f"{ref_id} - failed after multiple captcha attempts")
        
        # ⛔ If captcha failed totally, mark as failed
        pdf_info_list.append({
            "reference_id": ref_id,
            "pdf_name": "captcha failed / no pdf",
            "pdf_path": "captcha failed / no pdf"
            
        })

# Close browser only if you want right now i make it as comment
driver.quit()

GEM/2024/B/5588836 - no result found
GEM/2025/B/6071338 - successfully downloaded
GEM/2025/B/6071338 - PDF not found in folder!
GEM/2024/B/5594165 - captcha attempt 1 failed, retrying...
GEM/2024/B/5594165 - successfully downloaded
GEM/2024/B/5594165 - downloaded as GEMC-511687755678653-28112024.pdf
GEM/2024/B/5458607 - successfully downloaded
GEM/2024/B/5458607 - downloaded as GEMC-511687797936727-19122024.pdf
GEM/2024/B/5480361 - captcha attempt 1 failed, retrying...
GEM/2024/B/5480361 - successfully downloaded
GEM/2024/B/5480361 - downloaded as GEMC-511687753705287-01012025.pdf
GEM/2024/B/4560359 - successfully downloaded
GEM/2024/B/4560359 - downloaded as GEMC-511687772753116-22102024.pdf
GEM/2018/B/119037 - no result found
GEM/2022/B/1919736 - successfully downloaded
GEM/2022/B/1919736 - downloaded as GEMC-511687766920434-25022022.pdf
GEM/2024/B/5598504 - successfully downloaded
GEM/2024/B/5598504 - downloaded as GEMC-511687718974362-08012025.pdf


In [29]:
# Build DataFrame with all columns first
rdf = pd.DataFrame(pdf_info_list)

# Add hyperlink column using pdf_path
rdf["pdf_link"] = rdf["pdf_path"].apply(
    lambda x: f'=HYPERLINK("{x}", "Click Here")'
    if x not in ["no result found", "captcha failed / no pdf"] else x
)

# Keep only the 3 required columns
rdf = rdf[["reference_id", "pdf_name", "pdf_link"]]
print("all well")

all well


In [39]:
#shows the info
rdf.head(10)

,reference_id,pdf_name,pdf_link
0,GEM/2024/B/5588836,no result found,no result found
1,GEM/2025/B/6071338,no result found,no result found
2,GEM/2024/B/5594165,GEMC-511687755678653-28112024.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687755678653-28112024.pdf"", ""Click Here"")"
3,GEM/2024/B/5458607,GEMC-511687797936727-19122024.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687797936727-19122024.pdf"", ""Click Here"")"
4,GEM/2024/B/5480361,GEMC-511687753705287-01012025.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687753705287-01012025.pdf"", ""Click Here"")"
5,GEM/2024/B/4560359,GEMC-511687772753116-22102024.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687772753116-22102024.pdf"", ""Click Here"")"
6,GEM/2018/B/119037,no result found,no result found
7,GEM/2022/B/1919736,GEMC-511687766920434-25022022.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687766920434-25022022.pdf"", ""Click Here"")"
8,GEM/2024/B/5598504,GEMC-511687718974362-08012025.pdf,"=HYPERLINK(""C:\Users\admin\Downloads\gem_pdf\all_pdf\GEMC-511687718974362-08012025.pdf"", ""Click Here"")"


In [38]:
#from openpyxl import load_workbook

output_path = r"C:\Users\admin\Downloads\gem_pdf\winner_bidder_info.xlsx"
rdf.to_excel(output_path, index=False, engine="openpyxl")

# Open workbook to adjust formatting
wb = load_workbook(output_path)
ws = wb.active

# Set column widths
ws.column_dimensions["A"].width = 22     # reference_id
ws.column_dimensions["B"].width = 38.5   # pdf_name
ws.column_dimensions["C"].width = 18     # pdf_link

# Freeze header row
ws.freeze_panes = "A2"


# Style header row (A1, B1, C1)
header_font = Font(size=14, bold=True)
for cell in ws[1]:
    cell.font = header_font

# Apply borders to all cells
thin_border = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin")
)
for row in ws.iter_rows():
    for cell in row:
        cell.border = thin_border

# Center align last column (pdf_link)
for row in ws.iter_rows(min_row=2, min_col=3, max_col=3):
    for cell in row:
        cell.alignment = Alignment(horizontal="center")


wb.save(output_path)
print("Excel saved with formatting at:", output_path)

Excel saved with formatting at: C:\Users\admin\Downloads\gem_pdf\winner_bidder_info.xlsx
